<a href="https://colab.research.google.com/github/DogwonLee/Causal-Inference---Rent/blob/main/Rent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Step 2 - Clean & normalize the Evening Star OCR text and segment it into
rough "articles" (headline block + body block).

This is a rule-based cleaner (Option C style: regex/heuristics, no external
NLP libs needed since the sandbox has no network access for pip).

It is intentionally lightweight: the goal is not perfect OCR restoration,
but to (a) strip page markers, (b) repair hyphenation across line breaks,
(c) join OCR lines into readable paragraphs, and (d) split the issue into
headline+body chunks so Step 3 can find the rent-commission case articles.
"""

import re
import json
import glob
import os

PAGE_MARKER_RE = re.compile(r"^=====.*=====$")


def load_raw(path):
    with open(path, encoding="utf-8", errors="replace") as f:
        return f.read()


def split_pages(raw):
    """Return list of (page_number, page_text) using the '=== PAGE N ===' markers."""
    lines = raw.replace("\r\n", "\n").split("\n")
    pages = []
    current_page_num = 0
    current_lines = []
    for line in lines:
        m = re.match(r"=====\s*PAGE\s*(\d+)\s*=====", line.strip())
        if m:
            if current_lines:
                pages.append((current_page_num, current_lines))
            current_page_num = int(m.group(1))
            current_lines = []
            continue
        if PAGE_MARKER_RE.match(line.strip()):
            # the top "===== Evening Star - DATE =====" banner
            continue
        current_lines.append(line)
    if current_lines:
        pages.append((current_page_num, current_lines))
    return pages


def is_headline_line(line):
    """Heuristic: a line is part of a headline if it's short-ish and mostly
    uppercase letters (OCR headlines are set in caps)."""
    s = line.strip()
    if not (3 <= len(s) <= 90):
        return False
    letters = [c for c in s if c.isalpha()]
    if len(letters) < 3:
        return False
    upper_ratio = sum(c.isupper() for c in letters) / len(letters)
    return upper_ratio > 0.85


def join_body_lines(lines):
    """Join OCR body lines into a paragraph string, repairing end-of-line
    hyphenation (e.g. 'depriv-\\ned' -> 'deprived')."""
    out = ""
    for line in lines:
        s = line.strip()
        if not s:
            continue
        if out.endswith("-"):
            out = out[:-1] + s
        elif out:
            out = out + " " + s
        else:
            out = s
    return out


def segment_articles(pages):
    """Walk through each page's lines, grouping consecutive headline lines
    into a headline block, and the following non-headline lines into the
    body, until the next headline block starts a new article."""
    articles = []
    for page_num, lines in pages:
        headline_buf = []
        body_buf = []

        def flush():
            if headline_buf or body_buf:
                articles.append({
                    "page": page_num,
                    "headline": join_body_lines(headline_buf),
                    "body": join_body_lines(body_buf),
                })

        for line in lines:
            if is_headline_line(line):
                if body_buf:
                    # body lines ended -> previous article is complete
                    flush()
                    headline_buf, body_buf = [], []
                headline_buf.append(line)
            else:
                body_buf.append(line)
        flush()
    # drop empty/near-empty fragments
    return [a for a in articles if len(a["headline"]) + len(a["body"]) > 15]


KEYWORDS = [
    "rent commission", "rent law", "ball act", "ball rent",
    "petition", "determination", "increase rent", "reduce rent",
    "fair rent", "rents", "landlord", "tenant", "possession of",
]


def is_relevant(article):
    text = (article["headline"] + " " + article["body"]).lower()
    return any(kw in text for kw in KEYWORDS)


def main():
    in_dir = "/home/claude/rent_project/data"
    out_dir = "/home/claude/rent_project/output"
    os.makedirs(out_dir, exist_ok=True)

    summary = {}
    for path in sorted(glob.glob(os.path.join(in_dir, "1920-06-*.txt"))):
        date = os.path.basename(path).replace(".txt", "")
        raw = load_raw(path)
        pages = split_pages(raw)
        articles = segment_articles(pages)
        relevant = [a for a in articles if is_relevant(a)]

        # save full cleaned article list and the relevant subset
        with open(os.path.join(out_dir, f"{date}_articles.json"), "w", encoding="utf-8") as f:
            json.dump(articles, f, ensure_ascii=False, indent=2)
        with open(os.path.join(out_dir, f"{date}_relevant.json"), "w", encoding="utf-8") as f:
            json.dump(relevant, f, ensure_ascii=False, indent=2)

        summary[date] = {
            "total_articles": len(articles),
            "relevant_articles": len(relevant),
        }

    with open(os.path.join(out_dir, "clean_summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    for date, s in summary.items():
        print(f"{date}: {s['total_articles']} articles total, "
              f"{s['relevant_articles']} look rent-commission-related")


if __name__ == "__main__":
    main()